# <font color='red' size='6'>Inteligência Artificial</font>

# <font color='black' size='3'>Atividade: PLN - Parte I - Pré-processamento</font>

<b>Prática experimental na qual os alunos devem:</b><br>
<ul>
<li>preencher os dados abaixo da turma e integrantes com nome e RA</li>
<li>fornecer 10 novas mensagens de texto na lista de CENARIOS. CUIDADO: deve NECESSARIAMENTE fornecer o tipo de texto indicado em cada CENÁRIO</li> 
<li>executar este notebook</li> 
<li>gerar o arquivo texto “questionario_pln_10_textos_preenchido.md”</li> 
<li>preencher os dados da turma e dos integrantes no arquvo ".md" gerado, também com nome e RA, e</li> 
<li>responder as 13 perguntas que constam no arquivo gerado “questionario_pln_10_textos_preenchido.md”.</li> 
</ul>

#### Marcar com um X a turma a qual o grupo pertence.</br>

<b>Turma: ( &nbsp;&nbsp;&nbsp;&nbsp; )   7G</br> Turma: ( X ) 7N </b><p>
<table width="300" border="2">
    <tr>
        <td><b>Nome do Aluno</b> </td>
        <td><b>RA</b></td>
     </tr>  
   <tr>
       <td>Gabriel Ferreira </td>
       <td>10442043</td>
   </tr>
   <tr>
       <td>—</td>
       <td>—</td>
   </tr>
   <tr>
       <td>—</td>
       <td>—</td>
   </tr>
   <tr>
       <td>—</td>
       <td>—</td>
   </tr>
    <tr>
       <td>—</td>
       <td>—</td>
   </tr>
    <tr>
       <td>—</td>
       <td>—</td>
   </tr>
</table>

In [1]:
# Importa o módulo re (expressões regulares), utilizado para identificar,
# localizar, substituir e remover padrões em textos, como URLs, menções,
# pontuação, gírias e emojis.
import re

In [2]:
# Importa o módulo unicodedata, utilizado para normalizar caracteres Unicode,
# padronizando a representação de letras acentuadas (á, é, ã, ç etc.),
# sem necessariamente remover os acentos.
import unicodedata

In [3]:
# Importa a classe Path do módulo pathlib, utilizada para criar e manipular
# caminhos de arquivos e diretórios, permitindo salvar o questionário
# com os resultados do pré-processamento dos textos.
from pathlib import Path

In [4]:
GIRIAS = {
    "vc": "você", "vcs": "vocês", "pq": "porque", "tb": "também",
    "tbm": "também", "n": "não", "blz": "beleza", "mt": "muito",
    "vlw": "valeu", "msg": "mensagem", "mano": "amigo",
    "agr": "agora", "to": "estou", "vdd": "verdade", "hj": "hoje"
}

In [5]:
STOPWORDS = {
    "a", "as", "o", "os", "um", "uma", "uns", "umas", "de", "do", "da", "dos", "das",
    "em", "no", "na", "nos", "nas", "com", "por", "para", "e", "ou", "que",
    "é", "foi", "ser", "ao", "aos", "à", "às"
}

In [6]:
EMOJI = re.compile(
    r"[\U0001F300-\U0001F5FF\U0001F600-\U0001F64F"
    r"\U0001F680-\U0001F6FF\U0001F900-\U0001F9FF"
    r"\U0001FA70-\U0001FAFF\u2600-\u26FF\u2700-\u27BF]"
)

In [7]:
CENARIOS = [
    ('Maiúsculas e minúsculas', 'Na SEMANA passada estudei IA, mas HOJE revisei PLN.'),
    ('Gírias e abreviações', 'vc viu a msg? pq eu n recebi nada hj, vlw!'),
    ('Emojis e sentimento', 'A explicação foi ótima 🤩, porém o exercício me deixou confuso 😕.'),
    ('URL e menção', 'Consulte https://universidade.example/aula e avise @gabriel sobre o prazo.'),
    ('Hashtags e pontuação', '#AprendizadoDeMaquina: consegui terminar!!! E você???'),
    ('Acentos e Unicode', 'O café está ótimo; a ação da equipe foi rápida e eficiente.'),
    ('Stopwords', 'A professora apresentou o conteúdo para os alunos na sala de aula.'),
    ('Números e símbolos', 'Comprei 3 livros por R$ 120,50; o desconto foi de 15%.'),
    ('Negação e sentido', 'Eu não recomendo o curso: a aula não foi clara.'),
    ('Texto misto', 'MANO, vc leu a msg hj? 😅 Veja www.curso.example @turma #PLN!!!'),
]


In [8]:
# Converte o texto para minúsculas, normaliza seus caracteres Unicode
# e padroniza os espaços entre as palavras.
# Parâmetro: texto (str) - mensagem original.
# Retorno: str - mensagem normalizada.
# NFC recompõe caracteres Unicode equivalentes em forma canônica;
# não remove os acentos.
def normalizar(texto):
    return " ".join(unicodedata.normalize("NFC", texto).lower().split())

In [9]:
# Substitui gírias/abreviações cadastradas no dicionário GIRIAS.
# Parâmetro: texto (str) - texto preferencialmente já em minúsculas.
# Retorno: str - texto com abreviações reconhecidas substituídas.
# re.sub procura sequências de letras delimitadas por fronteiras de palavra (\b).
# Para cada correspondência, a função lambda consulta o dicionário;
# caso não encontre a palavra, preserva o texto original.
def expandir_girias(texto):
    return re.sub(r"\b[a-záàâãéêíóôõúüç]+\b",
                  lambda m: GIRIAS.get(m.group(0), m.group(0)), texto)

In [10]:
# Localiza caracteres emoji de acordo com a expressão regular EMOJI.
# Parâmetro: texto (str) - mensagem a inspecionar.
# Retorno: list[str] - ocorrências encontradas, na ordem do texto.
# Observação: detector simplificado, não cobre todos os emojis compostos.
def identificar_emojis(texto):
    return EMOJI.findall(texto)

In [11]:
# Remove elementos selecionados que não serão usados na análise textual.
# Parâmetro: texto (str) - mensagem a ser limpa.
# Retorno: str - texto com espaços padronizados após a limpeza.
# 1) Remove endereços web iniciados por http(s) ou www.
# 2) Remove menções iniciadas por @.
# 3) Substitui os emojis reconhecidos por espaços.
# 4) Remove pontuação e símbolos; hashtags perdem #, mas mantêm a palavra.
# 5) Converte sublinhados em espaços e elimina espaços repetidos.
# Números e letras acentuadas são mantidos.
def remover_nao_textuais(texto):
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = re.sub(r"(?<!\w)@[\w.]+", " ", texto)
    texto = EMOJI.sub(" ", texto)
    texto = re.sub(r"[^\w\sÀ-ÿ]", " ", texto)
    texto = texto.replace("_", " ")
    return " ".join(texto.split())

In [12]:
# Executa o fluxo completo de pré-processamento para uma mensagem.
# Parâmetro: texto (str) - texto fornecido pelo estudante.
# Retorno: dict - saídas intermediárias de todas as etapas.
# Os emojis são identificados no texto normalizado, pois a limpeza
# os removeria e impediria sua recuperação posterior.
# A tokenização didática é feita por split(), que divide por espaços.
# Stopwords são filtradas pela comparação com o conjunto STOPWORDS.
def processar(texto):
    # 1. Padronização de caixa, Unicode e espaços.
    normalizado = normalizar(texto)
    # 2. Expansão das abreviações conhecidas.
    expandido = expandir_girias(normalizado)
    # 3. Limpeza de URLs, menções, emojis e pontuação.
    sem_elementos = remover_nao_textuais(expandido)
    # 4. Separação das palavras por espaços.
    tokens = sem_elementos.split()
    # 5. Mantém apenas tokens ausentes da lista fixa de stopwords
    filtrados = [token for token in tokens if token not in STOPWORDS]
    return {
        "original": texto,
        "normalizacao": normalizado,
        "expansao": expandido,
        "limpeza": sem_elementos,
        "tokens": tokens,
        "sem_stopwords": filtrados,
        "emojis": identificar_emojis(normalizado),
    }

In [13]:
# Exibe na tela o texto e os resultados de cada etapa.
# Parâmetros: n (int) - número do texto; tipo (str) - característica;
# d (dict) - registro gerado pela função processar().
# Retorno: None - apenas imprime resultados no console.
# !r usa repr(), facilitando a visualização de listas e strings.
def imprimir_resultados(n, tipo, d):
    print(f"\n{'=' * 65}\nTEXTO {n:02d} — {tipo}\n{'=' * 65}")
    for titulo, chave in [
        ("0. Original", "original"), ("1. Minúsculas/normalização", "normalizacao"),
        ("2. Expansão de gírias", "expansao"), ("3. Remoção não textuais", "limpeza"),
        ("4. Tokenização", "tokens"), ("5. Sem stopwords", "sem_stopwords"),
        ("6. Emojis identificados", "emojis"),
    ]:
        print(f"{titulo}: {d[chave]!r}")
    print(f"Quantidade de tokens: {len(d['tokens'])} -> {len(d['sem_stopwords'])}")

In [14]:
# Monta e salva um questionário Markdown com os resultados dos 10 textos.
# Parâmetros: registros (list[dict]) - saídas de processar() para cada
# texto; caminho (Path) - caminho de destino do arquivo .md.
# Retorno: None - grava o questionário no disco em UTF-8.
# A lista linhas guarda os parágrafos e perguntas a serem salvos.
# zip(CENARIOS, registros) associa cada cenário ao resultado correspondente.
def criar_questionario(registros, caminho):
    linhas = [
        "# Atividade avaliativa — Pré-processamento em PLN (10 textos)", "",
        "**Turma:** 07N (atividade individual por exceção)", "",
        "**Nomes em ordem alfabética**", "",
        "**Nome 1:** __________________________________  **RA 1:** ______________", "",
        "**Nome 2:** __________________________________  **RA 2:** ______________", "",
        "**Nome 3:** __________________________________  **RA 3:** ______________", "",
        "**Nome 4:** __________________________________  **RA 4:** ______________", "",
        "**Nome 5:** __________________________________  **RA 5:** ______________", "",
        "**Nome 6:** __________________________________  **RA 6:** ______________", "",
        "## Orientações", "",
        "Informe dez textos, um por característica solicitada. Abaixo estão registradas as saídas",
        "do programa. Examine-as, responda às questões de cada texto e entregue este arquivo.",
        "As saídas são produzidas pelo código; a interpretação deve ser realizada pelo estudante.", "",
        "**Atenção:** a lista de stopwords e a identificação de emojis são simplificadas.",
        "A remoção de pontuação pode alterar significado; neste código, números são preservados.", ""
    ]
    questoes = [
        "Quais diferenças apareceram entre a mensagem original e a versão normalizada?",
        "Quais abreviações foram expandidas? Alguma deveria ter sido expandida e não foi?",
        "Quais emojis foram identificados? Que informação emocional pode se perder na limpeza?",
        "Quais informações da URL e da menção desapareceram? Isso seria sempre desejável?",
        "Como as hashtags e a pontuação foram tratadas? A palavra da hashtag foi mantida?",
        "O que ocorreu com os acentos? Seria conveniente removê-los neste problema?",
        "Quais stopwords foram removidas e quais permaneceram? Por quê?",
        "Como os números e símbolos monetários/percentuais foram tratados?",
        "A palavra 'não' foi preservada? Que risco haveria em removê-la?",
        "Quais etapas provocaram mudanças? Que informações relevantes foram perdidas?"
    ]
    for i, ((tipo, _), d) in enumerate(zip(CENARIOS, registros), 1):
        linhas += [f"## Texto {i:02d} — {tipo}", "", "**Resultados obtidos na execução:**", ""]
        for titulo, chave in [
            ("Texto original", "original"), ("1. Normalização", "normalizacao"),
            ("2. Expansão", "expansao"), ("3. Limpeza", "limpeza"),
            ("4. Tokens", "tokens"), ("5. Sem stopwords", "sem_stopwords"),
            ("6. Emojis", "emojis")
        ]:
            linhas.append(f"- **{titulo}:** `{d[chave]!r}`")
        linhas += [f"- **Número de tokens (antes → depois das stopwords):** {len(d['tokens'])} → {len(d['sem_stopwords'])}",
                   "", f"**Questão {i}:** {questoes[i-1]}", "", "**Resposta:**", "", "________________________________________________________________________", ""]
    linhas += [
        "## Questões finais", "",
        "**11.** Compare dois dos textos e indique uma transformação que pode prejudicar uma tarefa de análise de sentimentos. Justifique.",
        "", "**Resposta:**", "", "________________________________________________________________________", "",
        "**12.** Para utilizar BERTimbau, você removeria automaticamente stopwords, emojis e pontuação? Explique sua decisão.",
        "", "**Resposta:**", "", "________________________________________________________________________", "",
        "**13.** Em qual etapa um vazamento de informações entre treino e teste poderia ocorrer ao ajustar transformações aprendidas dos dados (como TF-IDF)?", "",
        "**Resposta:**", "", "________________________________________________________________________", ""
    ]
    caminho.write_text("\n".join(linhas), encoding="utf-8")

In [15]:
# Processa os dez textos próprios e gera o questionário sem entrada interativa.
def main():
    registros = []
    for i, (tipo, texto) in enumerate(CENARIOS, 1):
        resultado = processar(texto)
        registros.append(resultado)
        imprimir_resultados(i, tipo, resultado)
    destino = Path("questionario_pln_10_textos_preenchido.md")
    criar_questionario(registros, destino)
    print(f"Questionário criado em: {destino.resolve()}")


In [16]:
main()



TEXTO 01 — Maiúsculas e minúsculas
0. Original: 'Na SEMANA passada estudei IA, mas HOJE revisei PLN.'
1. Minúsculas/normalização: 'na semana passada estudei ia, mas hoje revisei pln.'
2. Expansão de gírias: 'na semana passada estudei ia, mas hoje revisei pln.'
3. Remoção não textuais: 'na semana passada estudei ia mas hoje revisei pln'
4. Tokenização: ['na', 'semana', 'passada', 'estudei', 'ia', 'mas', 'hoje', 'revisei', 'pln']
5. Sem stopwords: ['semana', 'passada', 'estudei', 'ia', 'mas', 'hoje', 'revisei', 'pln']
6. Emojis identificados: []
Quantidade de tokens: 9 -> 8

TEXTO 02 — Gírias e abreviações
0. Original: 'vc viu a msg? pq eu n recebi nada hj, vlw!'
1. Minúsculas/normalização: 'vc viu a msg? pq eu n recebi nada hj, vlw!'
2. Expansão de gírias: 'você viu a mensagem? porque eu não recebi nada hoje, valeu!'
3. Remoção não textuais: 'você viu a mensagem porque eu não recebi nada hoje valeu'
4. Tokenização: ['você', 'viu', 'a', 'mensagem', 'porque', 'eu', 'não', 'recebi', 'nada

### <font size = '2'>prof. Dr. Ivan Carlos Alcântara de Oliveira</font> - <font color="blue" size = '2'>https://orcid.org/0000-0002-6020-7535.</font><br><font size = '2'> e-mail: ivan.oliveira@mackenzie.br</font>